# 面试问题：DPO 如何直接利用 chosen/rejected 偏好训练策略模型？

可以直接复述的回答是：DPO 对同一 prompt 的 chosen 和 rejected 计算策略对数概率差，再减去参考模型的对数概率差。`β` 控制偏好优化相对参考策略的强度，目标是最大化 `log sigmoid(β*(policy_logratio-reference_logratio))`。参考项相当于隐式 KL 约束，防止策略无限偏离。实现时 chosen/rejected 顺序、mask 和长度归一化必须一致；符号写反会稳定地优化错误偏好。下面用八组可读客服回答特征，手写 pair log-prob、DPO loss 和真实参数更新。

## 真实案例：选择更有帮助、事实明确且安全的客服回复

每个回答用四个脱敏属性表示：帮助性、事实性、安全性和冗长度。八个 prompt 都给出 chosen/rejected 文本摘要；特征只是教学 reward 表示，不是线上大模型 token 概率。

In [1]:
import warnings  # 导入告警控制模块
import torch  # 导入 PyTorch 张量和自动微分
warnings.filterwarnings("ignore")  # 隐藏环境告警保持输出清晰
torch.set_num_threads(1)  # 固定 CPU 单线程执行
torch.manual_seed(1302)  # 固定策略初始化
pairs = [  # 定义八组带语义的偏好数据
    ("D-01", "退款多久到账", "说明3至5个工作日并给查询入口", "保证今天一定到账", [0.9, 0.9, 0.9, 0.2], [0.4, 0.1, 0.3, 0.1]),  # 事实准确退款答复
    ("D-02", "包裹丢失怎么办", "先核验物流并发起赔付流程", "让用户自行联系快递", [0.9, 0.8, 0.9, 0.3], [0.2, 0.5, 0.5, 0.1]),  # 主动处理物流问题
    ("D-03", "账户疑似被盗", "冻结高风险操作并引导身份核验", "直接发送用户全部账户信息", [0.8, 0.8, 1.0, 0.3], [0.5, 0.4, 0.0, 0.2]),  # 安全账户答复
    ("D-04", "设备无法开机", "给出电源和复位两步排查", "建议立刻购买新设备", [0.9, 0.8, 0.8, 0.2], [0.2, 0.4, 0.6, 0.1]),  # 可操作技术排查
    ("D-05", "能否修改地址", "说明发货前可修改并给操作路径", "只回复可以", [0.8, 0.9, 0.9, 0.2], [0.4, 0.6, 0.8, 0.0]),  # 完整地址修改答复
    ("D-06", "优惠券失效", "核对有效期并说明补偿条件", "承诺无条件补发十张", [0.8, 0.9, 0.8, 0.3], [0.5, 0.1, 0.4, 0.1]),  # 不虚假承诺
    ("D-07", "如何删除数据", "说明身份验证与删除时限", "要求用户公开身份证照片", [0.8, 0.8, 1.0, 0.3], [0.3, 0.4, 0.0, 0.2]),  # 隐私安全答复
    ("D-08", "会员自动续费", "解释关闭路径与已扣款申诉入口", "用很长营销话术回避问题", [0.9, 0.8, 0.9, 0.2], [0.2, 0.3, 0.5, 1.0]),  # 简洁直接答复
]  # 结束八组偏好样本
chosen_features = torch.tensor([row[4] for row in pairs], dtype=torch.float32)  # 构造 chosen 四维特征
rejected_features = torch.tensor([row[5] for row in pairs], dtype=torch.float32)  # 构造 rejected 四维特征
reference_weight = torch.tensor([0.35, 0.30, 0.45, -0.05], dtype=torch.float32)  # 定义冻结参考策略特征权重
print("输入预览：id | prompt | chosen | rejected")  # 输出偏好数据表头
for row in pairs:  # 逐组展示八个 prompt
    print(f"{row[0]} | {row[1]:8} | {row[2]} | {row[3]}")  # 展示可读 chosen/rejected
print("特征顺序：[帮助性, 事实性, 安全性, 冗长度]")  # 解释模型输入含义

输入预览：id | prompt | chosen | rejected
D-01 | 退款多久到账   | 说明3至5个工作日并给查询入口 | 保证今天一定到账
D-02 | 包裹丢失怎么办  | 先核验物流并发起赔付流程 | 让用户自行联系快递
D-03 | 账户疑似被盗   | 冻结高风险操作并引导身份核验 | 直接发送用户全部账户信息
D-04 | 设备无法开机   | 给出电源和复位两步排查 | 建议立刻购买新设备
D-05 | 能否修改地址   | 说明发货前可修改并给操作路径 | 只回复可以
D-06 | 优惠券失效    | 核对有效期并说明补偿条件 | 承诺无条件补发十张
D-07 | 如何删除数据   | 说明身份验证与删除时限 | 要求用户公开身份证照片
D-08 | 会员自动续费   | 解释关闭路径与已扣款申诉入口 | 用很长营销话术回避问题
特征顺序：[帮助性, 事实性, 安全性, 冗长度]


## Baseline / 基线：冻结参考策略的偏好 margin

将两个回答看作当前 prompt 下的二选一，pair log-prob 差等于两个 response score 的差。先观察参考策略 margin 与 pair accuracy。

In [2]:
class ResponsePolicy(torch.nn.Module):  # 定义最小可训练回答打分策略
    def __init__(self, initial_weight):  # 初始化四维策略权重
        super().__init__()  # 初始化 PyTorch 模块基类
        self.weight = torch.nn.Parameter(initial_weight.clone())  # 注册可训练特征权重
    def forward(self, chosen, rejected):  # 为同 prompt 两个回答计算归一化对数概率
        chosen_score = chosen @ self.weight  # 计算 chosen 未归一化分数
        rejected_score = rejected @ self.weight  # 计算 rejected 未归一化分数
        stacked = torch.stack([chosen_score, rejected_score], dim=1)  # 组成每个 prompt 的二回答 logits
        shifted = stacked - stacked.max(dim=1, keepdim=True).values  # 数值稳定地平移 logits
        log_probability = shifted - torch.log(torch.exp(shifted).sum(dim=1, keepdim=True))  # 手写二项 log-softmax
        return log_probability[:, 0], log_probability[:, 1]  # 返回 chosen 与 rejected 对数概率
reference_model = ResponsePolicy(reference_weight)  # 创建参考策略模型
reference_model.weight.requires_grad_(False)  # 冻结参考权重避免被 DPO 更新
with torch.no_grad():  # 禁止参考策略建立训练计算图
    reference_chosen, reference_rejected = reference_model(chosen_features, rejected_features)  # 计算八组参考 log-prob
    reference_margin = reference_chosen - reference_rejected  # 计算参考策略偏好 margin
baseline_pair_accuracy = float((reference_margin > 0.0).float().mean())  # 计算参考策略选择 chosen 的比例
print("id | ref_logp_chosen | ref_logp_rejected | margin | correct")  # 输出参考策略逐 pair 表头
for index, row in enumerate(pairs):  # 遍历八组偏好
    print(f"{row[0]} | {reference_chosen[index]:.4f} | {reference_rejected[index]:.4f} | {reference_margin[index]:.4f} | {bool(reference_margin[index] > 0)}")  # 展示参考偏好强度
print(f"参考策略 pair accuracy={baseline_pair_accuracy:.1%}")  # 汇总冻结基线

id | ref_logp_chosen | ref_logp_rejected | margin | correct
D-01 | -0.4099 | -1.0899 | 0.6800 | True
D-02 | -0.4722 | -0.9772 | 0.5050 | True
D-03 | -0.4132 | -1.0832 | 0.6700 | True
D-04 | -0.4932 | -0.9432 | 0.4500 | True
D-05 | -0.5694 | -0.8344 | 0.2650 | True
D-06 | -0.4684 | -0.9834 | 0.5150 | True
D-07 | -0.3901 | -1.1301 | 0.7400 | True
D-08 | -0.4322 | -1.0472 | 0.6150 | True
参考策略 pair accuracy=100.0%


## 核心实现：策略 logratio、参考 logratio 与 DPO loss

策略从参考权重初始化。每一步都重新计算八组 pair，对 DPO loss backward，再手写更新四维权重。

In [3]:
def dpo_loss(policy_chosen, policy_rejected, ref_chosen, ref_rejected, beta=0.5, reverse=False):  # 手写 DPO 二元偏好目标
    policy_logratio = policy_chosen - policy_rejected  # 计算策略 chosen 相对 rejected 对数比
    reference_logratio = ref_chosen - ref_rejected  # 计算参考策略对数比
    advantage = policy_logratio - reference_logratio  # 形成相对参考策略的隐式奖励差
    signed_advantage = -advantage if reverse else advantage  # 允许复现 chosen/rejected 符号写反
    losses = torch.log1p(torch.exp(-beta * signed_advantage))  # 用稳定 softplus 计算负 log-sigmoid
    return losses.mean(), advantage, losses  # 返回平均损失、相对优势和逐 pair 损失
policy_model = ResponsePolicy(reference_weight)  # 从参考权重创建待优化策略
training_trace = []  # 保存关键步损失、梯度和 margin
for step in range(1, 121):  # 执行一百二十步全批量偏好优化
    policy_chosen, policy_rejected = policy_model(chosen_features, rejected_features)  # 真实执行策略 forward
    loss, advantage, pair_losses = dpo_loss(policy_chosen, policy_rejected, reference_chosen, reference_rejected)  # 计算参考约束 DPO 目标
    loss.backward()  # 真实执行 backward 得到四维策略梯度
    gradient = policy_model.weight.grad.detach().clone()  # 保存当前权重梯度
    with torch.no_grad():  # 关闭手写更新的计算图
        policy_model.weight -= 0.25 * policy_model.weight.grad  # 沿 DPO 负梯度更新策略
        policy_model.weight.grad.zero_()  # 清空本步梯度
    if step in {1, 2, 20, 120}:  # 保存关键偏好训练节点
        training_trace.append((step, float(loss), float(advantage.mean()), gradient.clone(), policy_model.weight.detach().clone()))  # 记录损失、相对优势和权重
with torch.no_grad():  # 进入策略评估阶段
    final_chosen, final_rejected = policy_model(chosen_features, rejected_features)  # 计算训练后 pair log-prob
    final_margin = final_chosen - final_rejected  # 计算最终 chosen margin
final_pair_accuracy = float((final_margin > 0.0).float().mean())  # 计算最终 pair accuracy
print("step | DPO loss | mean relative advantage | gradient | weight")  # 输出 DPO 训练轨迹表头
for item in training_trace:  # 遍历四个关键节点
    print(f"{item[0]:4d} | {item[1]:.5f} | {item[2]:.5f} | {[round(value, 4) for value in item[3].tolist()]} | {[round(value, 3) for value in item[4].tolist()]}")  # 展示真实参数更新
print(f"DPO pair accuracy={final_pair_accuracy:.1%}")  # 汇总训练后偏好命中

step | DPO loss | mean relative advantage | gradient | weight
   1 | 0.69315 | 0.00000 | [-0.1281, -0.1219, -0.1281, -0.0063] | [0.382, 0.33, 0.482, -0.048]
   2 | 0.68129 | 0.04772 | [-0.1266, -0.1204, -0.1264, -0.0062] | [0.414, 0.361, 0.514, -0.047]
  20 | 0.51152 | 0.81226 | [-0.1026, -0.0966, -0.0997, -0.0055] | [0.924, 0.843, 1.016, -0.021]
 120 | 0.19249 | 3.20266 | [-0.0443, -0.0399, -0.038, -0.0036] | [2.576, 2.367, 2.531, 0.089]
DPO pair accuracy=100.0%


## 结果表：八组 margin 如何变化

In [4]:
print("id | ref_margin | policy_margin | relative_gain | preferred")  # 输出逐 pair 对照表头
margin_gains = []  # 收集相对参考策略 margin 增量
for index, row in enumerate(pairs):  # 遍历八组客服偏好
    gain = float(final_margin[index] - reference_margin[index])  # 计算 DPO 相对参考策略增量
    margin_gains.append(gain)  # 保存逐样本偏好改善
    print(f"{row[0]} | {reference_margin[index]:10.4f} | {final_margin[index]:13.4f} | {gain:13.4f} | {bool(final_margin[index] > 0)}")  # 展示策略如何增加 chosen 优势
print(f"平均 margin：reference={float(reference_margin.mean()):.4f}，policy={float(final_margin.mean()):.4f}")  # 汇总偏好强度变化

id | ref_margin | policy_margin | relative_gain | preferred
D-01 |     0.6800 |        4.7085 |        4.0285 | True
D-02 |     0.5050 |        3.5432 |        3.0382 | True
D-03 |     0.6700 |        4.2589 |        3.5889 | True
D-04 |     0.4500 |        3.2649 |        2.8149 | True
D-05 |     0.2650 |        2.0112 |        1.7462 | True
D-06 |     0.5150 |        3.6961 |        3.1811 | True
D-07 |     0.7400 |        4.7741 |        4.0341 | True
D-08 |     0.6150 |        3.9277 |        3.3127 | True
平均 margin：reference=0.5550，policy=3.7731


## 失败案例与修正：chosen/rejected 符号写反

错误实现仍能稳定降低它自己的 loss，但会把策略推向 rejected。我们从同一参考初值各做一步，直接比较更新后平均 margin。

In [5]:
def single_dpo_step(reverse):  # 从参考初值执行一次正确或反向 DPO 更新
    probe = ResponsePolicy(reference_weight)  # 创建公平比较的策略探针
    chosen_logp, rejected_logp = probe(chosen_features, rejected_features)  # 对八组 pair 执行 forward
    loss, advantage, losses = dpo_loss(chosen_logp, rejected_logp, reference_chosen, reference_rejected, reverse=reverse)  # 按指定符号计算目标
    loss.backward()  # 真实执行首步 backward
    with torch.no_grad():  # 关闭探针参数更新计算图
        probe.weight -= 0.50 * probe.weight.grad  # 用相同步长更新正确或错误策略
        updated_chosen, updated_rejected = probe(chosen_features, rejected_features)  # 计算更新后 pair log-prob
    return float(loss), float((updated_chosen - updated_rejected).mean()), probe.weight.grad.detach().clone()  # 返回损失、平均 margin 和梯度
correct_loss, correct_one_step_margin, correct_gradient = single_dpo_step(False)  # 执行正确 chosen 优先更新
reversed_loss, reversed_one_step_margin, reversed_gradient = single_dpo_step(True)  # 复现符号写反更新
reference_mean_margin = float(reference_margin.mean())  # 读取两种探针共享起点 margin
print(f"起点 margin={reference_mean_margin:.5f}")  # 展示公平比较起点
print(f"正确符号：loss={correct_loss:.5f}，更新后 margin={correct_one_step_margin:.5f}，grad={correct_gradient.tolist()}")  # 展示 chosen margin 上升
print(f"反向符号：loss={reversed_loss:.5f}，更新后 margin={reversed_one_step_margin:.5f}，grad={reversed_gradient.tolist()}")  # 展示 rejected margin 被错误增强

起点 margin=0.55500
正确符号：loss=0.69315，更新后 margin=0.65045，grad=[-0.12812501192092896, -0.12187501043081284, -0.12812501192092896, -0.006250001490116119]
反向符号：loss=0.69315，更新后 margin=0.45955，grad=[0.12812501192092896, 0.12187501043081284, 0.12812501192092896, 0.006250001490116119]


## 结果解读

DPO 不是直接最大化 chosen 的绝对分数，而是优化相对参考策略的 chosen/rejected logratio。逐 pair 表显示八组 margin 都增加。符号反例尤其危险，因为错误目标的 loss 也能下降，只有 preference margin 和人工样本检查才能发现方向错误。

## 生产边界

教学策略是四特征线性打分，不含 token mask、序列长度和大模型归一化。生产 DPO 要用冻结 reference checkpoint，正确求和 response token log-prob，屏蔽 prompt/padding，并监控 KL、长度偏置、win rate 与安全切片。偏好数据可能有标注者偏差，beta、学习率和数据混合需在留出集选择。

## 最小回归测试

In [6]:
assert len(pairs) >= 6  # 保证偏好训练包含多个真实语义样本
assert training_trace[-1][1] < training_trace[0][1]  # 保证真实 DPO 更新降低目标损失
assert final_pair_accuracy >= baseline_pair_accuracy  # 保证策略偏好命中不低于参考基线
assert all(gain > 0.0 for gain in margin_gains)  # 保证八组 chosen margin 均相对参考增加
assert correct_one_step_margin > reference_mean_margin  # 保证正确符号首步推动 chosen
assert reversed_one_step_margin < reference_mean_margin  # 保证反向符号失败真实推动 rejected
assert torch.allclose(correct_gradient, -reversed_gradient)  # 保证同起点下错误符号产生相反梯度